# 04.8 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/04-trustworthy-models/08-pipelines/exercises.ipynb)

## Exercise 1: by hand

Training values `[4, 6, 8, 10]`, test value `2`. Compute the test value's z-score with the scaler fitted (a) on training only and (b) on all five values. Use the population standard deviation (divide by n). Check in NumPy.

In [ ]:
# your code here

In [ ]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

raw = load_retail("Year 2010-2011")                       # ~1 s after the first run
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

cutoff = pd.Timestamp("2011-09-01")                       # features before, target after
past = sales[sales["InvoiceDate"] < cutoff]
future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < "2011-12-01")]

cust = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
                                       products=("StockCode", "nunique"),
                                       first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
cust["tenure_days"] = (cutoff - cust["first_order"]).dt.days
cust["days_since_last"] = (cutoff - cust["last_order"]).dt.days
future_spend = future.groupby("Customer ID")["Revenue"].sum().reindex(cust.index).fillna(0)
cust["churned"] = (future_spend == 0).astype(int)        # 1 = did not come back
cust = cust.drop(columns=["first_order", "last_order"])
features = ["orders", "spend", "products", "tenure_days", "days_since_last"]
print(f"{len(cust):,} customers, churn rate {cust['churned'].mean():.1%}")

cust["avg_basket"] = cust["spend"] / cust["orders"]
cust["orders_per_month"] = cust["orders"] / (cust["tenure_days"] / 30).clip(lower=1)
cust["log_spend"] = np.log1p(cust["spend"])
country = past.groupby("Customer ID")["Country"].agg(lambda s: s.mode()[0])
cust["country"] = country.reindex(cust.index)
cust["is_uk"] = (cust["country"] == "United Kingdom").astype(int)
weekend = past.assign(wk=past["InvoiceDate"].dt.dayofweek >= 5).groupby("Customer ID")["wk"].mean()
cust["share_weekend"] = weekend.reindex(cust.index)
autumn = past[past["InvoiceDate"] >= "2011-06-01"].groupby("Customer ID")["Invoice"].nunique()
cust["orders_last_3m"] = autumn.reindex(cust.index).fillna(0)

from sklearn.model_selection import train_test_split
num_cols = ["orders", "spend", "products", "tenure_days", "days_since_last", "avg_basket"]
cat_cols = ["country"]
train, test = train_test_split(cust, test_size=0.25, random_state=42, stratify=cust["churned"])
X_train, y_train = train[num_cols + cat_cols], train["churned"]
X_test, y_test = test[num_cols + cat_cols], test["churned"]
print(f"train {len(X_train):,} / test {len(X_test):,}")

## Exercise 2: swap the model

Replace `LogisticRegression` in the pipeline with `KNeighborsClassifier(n_neighbors=101)`. Cross-validate. Then remove the scaler from the numeric branch and cross-validate again. What happens, and why?

In [ ]:
# your code here

## Exercise 3: a mean-imputation leak?

Copy `X_train` and blank out 30% of `spend` at random. Compare CV AUC when the imputer is fitted on all of `X_train` first versus inside the pipeline. Is the leak big or small? Why?

In [ ]:
# your code here

## Exercise 4: names after the one-hot

With `set_output(transform="pandas")`, list which countries got their own column and how many customers fell into the infrequent bucket.

In [ ]:
# your code here